# Phase 0: Setup
Setting up FastF1 where we will be gathering our data for the project

In [1]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

fastf1.Cache.enable_cache("../cache/")

# Phase 1: EDA: Bahrain Grand Prix 2023 analysis


We are starting by analyzing one race, preferably in a race where the circuit is pretty stable with overtaking opportunities, balanced pace etc. One of the choices is the Bahrain circuit, it's special because it's where tyre degradation appears to be and there are lots of overtakings, one of the reasons why it's the pre-season testing circuit so it's where teams actually test their cars before the season begins.

We load and analyze the race data

In [2]:
bahrain2023_race = fastf1.get_session(2023, "Bahrain", "R")
bahrain2023_race.load()
laps = bahrain2023_race.laps

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


In [3]:
print(laps.shape)
print(laps.columns)
print(laps.dtypes)
print(laps.head())


(1056, 31)
Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate'],
      dtype='object')
Time                  timedelta64[ns]
Driver                         object
DriverNumber                   object
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time           timedelta64[ns]
Sector1SessionTime    timedelta64[ns]
Sector2SessionTime    timedel

## Feature Engineering and cleaning

Before going forward we have to clean our data, there are laps that don't count, either in/out laps, laps under safety car, invalid laps etc and we need to fixate on one driver to actually have a consistent experiment piece

In [4]:
laps["LapTimeSeconds"] = laps["LapTime"].dt.total_seconds()
laps["Stint"] = laps["Stint"].astype(int)
clean_laps = laps[laps["IsAccurate"] == True]
drivers = clean_laps["Driver"].unique()
for d in drivers:
    ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
    print("Driver: ", d)
    print(ds)
    for s in ds:
        print(f"stint {s}: ", clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)].shape[0])

Driver:  VER
[1 2 3]
stint 1:  12
stint 2:  20
stint 3:  18
Driver:  GAS
[1 2 3 4]
stint 1:  7
stint 2:  14
stint 3:  13
stint 4:  16
Driver:  PER
[1 2 3]
stint 1:  15
stint 2:  15
stint 3:  20
Driver:  ALO
[1 2 3]
stint 1:  12
stint 2:  18
stint 3:  21
Driver:  LEC
[1 2 3]
stint 1:  11
stint 2:  18
stint 3:  5
Driver:  STR
[1 2 3]
stint 1:  13
stint 2:  13
stint 3:  24
Driver:  SAR
[1 2 3 4]
stint 1:  10
stint 2:  16
stint 3:  8
stint 4:  15
Driver:  MAG
[1 2 3 4]
stint 1:  13
stint 2:  12
stint 3:  9
stint 4:  15
Driver:  DEV
[1 2 3]
stint 1:  9
stint 2:  14
stint 3:  26
Driver:  TSU
[1 2 3 4]
stint 1:  8
stint 2:  14
stint 3:  12
stint 4:  16
Driver:  ALB
[1 2 3 4]
stint 1:  9
stint 2:  13
stint 3:  12
stint 4:  16
Driver:  ZHO
[1 2 3 4]
stint 1:  10
stint 2:  18
stint 3:  18
stint 4:  1
Driver:  HUL
[1 2 3 4]
stint 1:  9
stint 2:  13
stint 3:  12
stint 4:  15
Driver:  OCO
[1 2 3 4]
stint 1:  10
stint 2:  1
stint 3:  15
stint 4:  6
Driver:  NOR
[1 2 3 4 5 6]
stint 1:  8
stint 2:  5


In [6]:
for d in drivers:
    ds = clean_laps[clean_laps["Driver"] == d].Stint.unique()
    for s in ds:
        driver_stint = clean_laps[(clean_laps["Driver"] == d) & (clean_laps["Stint"] == s)]
        for i in driver_stint.index:
            #print(f"Lap: {driver_stint.loc[i, 'LapNumber'].astype(int)} Tyre age: {driver_stint.loc[i, 'TyreLife'].astype(int)} Laptime: {driver_stint.loc[i, 'LapTimeSeconds']}")
            continue
        plt.xlabel("Tyre Age")
        plt.ylabel("Lap Time")
        plt.title(f"{d} Stint {s} Compound: {driver_stint.loc[i, 'Compound']} Bahrain 2023 laptime evolution by tyre age (Lower is faster)")
        plt.plot(np.array(driver_stint['TyreLife'].astype(int)), np.array(driver_stint['LapTimeSeconds']))
        plt.savefig(f"../figs/bahrain2023_{d}_stint{s}_{driver_stint.loc[i, 'Compound']}_tyredeg_evo.png")
        plt.cla()
plt.close()

We need to analyze the fuel effect in the results, but before that we need to get the coefficient of how many seconds we save per kg of fuel burned, we do the analysis on one driver:

In [32]:
from sklearn.linear_model import LinearRegression
START_FUEL = 109.0
FUEL_BURN = START_FUEL/57
for d in drivers:
    driver_laps = clean_laps[clean_laps["Driver"] == d].copy()
    if driver_laps["LapNumber"].iloc[-1] < 50.0 :
        continue
    y = driver_laps[['LapTimeSeconds']]
    driver_laps["FuelMass"] = START_FUEL - (FUEL_BURN * driver_laps["LapNumber"]) + 1
    X = driver_laps[["TyreLife", "FuelMass"]]
    laptime_predictor = LinearRegression()
    laptime_predictor.fit(X, y)
    print(f"Driver: {d}, deg rate = {laptime_predictor.coef_[0][0]}, y ={laptime_predictor.coef_[0][1]}")

Driver: VER, deg rate = 0.048257834156160195, y =0.019029159102707114
Driver: GAS, deg rate = 0.17108125173964497, y =0.05178181037590402
Driver: PER, deg rate = 0.07864057877440597, y =0.02799122138024435
Driver: ALO, deg rate = 0.10825465860331517, y =0.040848078000197546
Driver: STR, deg rate = 0.12740757826327953, y =0.04362048864341722
Driver: SAR, deg rate = 0.16063828183575815, y =0.04256943466319926
Driver: MAG, deg rate = 0.21660975895643664, y =0.04843709506863638
Driver: DEV, deg rate = 0.17742970515828793, y =0.0512165617113019
Driver: TSU, deg rate = 0.15736704066491417, y =0.0480639700806829
Driver: ALB, deg rate = 0.14649786080667046, y =0.039312215177561664
Driver: ZHO, deg rate = 0.17722875335564964, y =0.05725646485829636
Driver: HUL, deg rate = 0.22237779563549143, y =0.0508816355942428
Driver: NOR, deg rate = 0.06723670405521133, y =0.045787526346744746
Driver: HAM, deg rate = 0.09987329656507728, y =0.034575071657724846
Driver: SAI, deg rate = 0.09351501233942466, 